# 02 — Modeling: Prediksi Posisi Grid Kualifikasi

Notebook ini membangun model untuk memprediksi **`grid_position`** (posisi grid hasil kualifikasi)
dari data sebelum kualifikasi (FP1–FP3).

**Input (fitur):** pace FP, kecepatan, cuaca  
**Output (target):** `grid_position` (1 = pole, 20 = terakhir)

Prasyarat: `01_tables.ipynb` sudah dijalankan sehingga `data/driver_quali.csv` ada.
Notebook ini **hanya membaca CSV** — tidak membaca parquet mentah.

## Pengenalan — Apa yang diprediksi, pakai model apa, dan kenapa ada baseline?

### 1. Yang diprediksi

Satu target utama: **`grid_position`** — posisi seorang pembalap di grid kualifikasi.

```text
Fitur FP (sebelum quali)          Prediksi
─────────────────────────         ─────────────────
fp_best_gap       ─┐
fp_best_rank       │             grid_position = 4
fp_median_gap      ├── model ─>   (artinya: prediksi grid ke-4)
```

### 1b. Kenapa fiturnya berupa **selisih (gap/rank)**, bukan angka mentah?

Target `grid_position` itu **relatif dalam 1 weekend** (urutan di GP tersebut).
Sebaliknya, angka lap mentah **absolut** dan bervariasi antar sirkuit
(Monza ±79 detik, Spa ±106 detik — rentang 39 detik).

Model yang dilatih pada angka absolut justru belajar **mengenali sirkuit**,
bukan mengenali pembalap tercepat:

```text
Salah  : Verstappen di Monaco (72.0s) vs Leclerc di Monza (79.0s)
        → model bilang Verstappen menang, padahal beda sirkuit!

Benar  : di Monza saja → Leclerc 79.0 vs Norris 79.5 → Leclerc lebih cepat
        → perbandingan dalam sirkuit yang sama (itulah aturan baseline)
```

**Bukti dari data sendiri** (korelasi `fp_best_lap` → `grid_position`):

| Cara hitung | Korelasi | Kesimpulan |
|---|---|---|
| Digabung semua sirkuit (**pool** = cara model belajar dulu) | **0.061** | nyaris nol — tidak ada sinyal |
| Diukur **per weekend** (cara baseline) | **0.657** | kuat — sinyal jelas |

Maka fitur diubah ke bahasa yang sama dengan target:

| Fitur baru | Arti |
|---|---|
| `fp_best_gap` | Selisih detik ke lap tercepat **di weekend yang sama** (0 = tercepat) |
| `fp_best_rank` | Peringkat dalam weekend (1 = tercepat) = **persis informasi baseline** |
| `fp_median_gap` | Selisih pace konsisten (median) ke tercepat — menangkap cepat-stabil vs 1 lap magis |

Karena `fp_best_rank` memuat aturan baseline, model **paling tidak bisa meniru baseline**;
ditambah `fp_best_gap` + `fp_median_gap` (informasi lebih kaya), model berpotensi
**mengalahkan** baseline. Fitur yang dibuang: angka absolut (`fp_best_lap`,
`fp_median_clean_lap`, `fp_avg_st_speed`) dan cuaca (sama untuk semua pembalap
dalam 1 weekend → tidak bisa membedakan urutan grid).

### 1c. Lalu kenapa perlu **fitur histori**?

Setelah fitur relatif dipasang, model sudah jauh lebih baik tapi **belum mengalahkan
baseline**. Penyebabnya: ada hal yang **tidak terlihat dari 1 weekend latihan** —
kualifikasi punya faktor knock-out Q1/Q2/Q3, track evolution, dan tow/slipstream.

Solusinya menambah ingatan tentang performa **lintas weekend**:

| Fitur histori | Arti |
|---|---|
| `hist_grid_season` | Rata-rata grid pembalap di weekend **sebelumnya** — pembalap top konsisten dapat grid depan |
| `hist_n` | Berapa weekend sudah dilewati (debut = kosong, wajar bila kurang stabil) |

**Anti-leakage:** dihitung dengan `sort chronological` + `expanding().mean().shift(1)` —
tiap baris hanya memakai grid dari weekend yang **sudah lewat**, tidak pernah memakai
grid weekend berjalan apalagi sesi kualifikasi. (Verifikasi dilakukan di notebook:
rata-rata histori dibandingkan ulang dengan data murni masa lalu.)

**Top-3 dan top-10 bukan model terpisah** — hanya turunan dari prediksi `grid_position`:

```text
prediksi grid = 4  →  is_top3 = TIDAK (4 > 3),  is_top10 = YA (4 ≤ 10)
```

### 2. Model yang dipakai

| Model | Tipe | Kenapa dicoba |
|---|---|---|
| **Ridge** | Regresi linear | Paling sederhana, jarang overfit, jadi patokan minimal |
| **Random Forest** | Regresi tree | Menangkap hubungan non-linear antara pace FP dan grid |
| **HistGradientBoosting** | Boosting | Biasanya paling akurat kalau data cukup banyak |
| **XGBoost** | Boosting | Kandidat modern; library dipasang untuk eksperimen lanjutan |

Keempatnya memprediksi **hal yang sama** (angka `grid_position`), jadi bukan 4 tujuan berbeda —
melainkan 4 kandidat yang **dibandingkan di validasi** untuk memilih yang terbaik.

### 3. Kenapa ada baseline?

**Baseline bukan model machine learning.** Baseline adalah aturan sederhana:

```text
Baseline = urutkan pembalap menurut fp_best_lap tercepat → itulah prediksi grid-nya
```

Fungsinya sebagai **pengukur keberhasilan**. Kalau model canggih tidak bisa
mengalahkan aturan sederhana ini, berarti modelnya tidak memberi nilai tambah
(fiturnya kurang, bukan modelnya yang kurang canggih).

### 4. Hubungan antar ketiganya

```text
Baseline  ←─ dibandingkan ─→  Model (Ridge / RF / HistGB)
                              │
                              └── prediksi grid_position
                                    ├── turunan: is_top3
                                    └── turunan: is_top10
```

- **Baseline ↔ Model:** perbandingan kinerja (model harus menang)
- **Model ↔ is_top3/is_top10:** sebab-akibat (klasifikasi **diturunkan** dari angka regresi, bukan model independen)

## 0. Setup & muat data

Membaca `config.yaml` dan `data/driver_quali.csv`.

Output: shape tabel, daftar tahun, jumlah weekend.

In [1]:
import numpy as np
import pandas as pd
import yaml
import json
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, roc_auc_score, ndcg_score
from sklearn.inspection import permutation_importance
from scipy.stats import spearmanr, kendalltau
from xgboost import XGBRegressor

# Cari root project. Aman dijalankan dari folder notebooks/ maupun dari root.
ROOT = Path.cwd()
if not (ROOT / "config.yaml").exists() and (ROOT.parent / "config.yaml").exists():
    ROOT = ROOT.parent

with (ROOT / "config.yaml").open("r", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle) or {}


SEED = cfg.get("model", {}).get("random_seed", 42)


def resolve_path(value):
    path = Path(value)
    return path if path.is_absolute() else ROOT / path


driver_quali_path = resolve_path(cfg["paths"]["driver_quali"])
df = pd.read_csv(driver_quali_path)

print("driver_quali:", df.shape)
print("Tahun:", sorted(df["year"].unique()))
print("Weekend unik:", df["meeting_key"].nunique())
df.head()

driver_quali: (1974, 21)
Tahun: [np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Weekend unik: 91


,meeting_key,driver_number,fp_best_lap,fp_median_clean_lap,fp_avg_st_speed,fp_best_gap,fp_best_rank,fp_median_gap,air_temperature,track_temperature,...,wind_speed,rainfall,grid_position,quali_lap_duration,year,circuit_short_name,is_top3,is_top10,hist_grid_season,hist_n
0,1141,1,91.076,98.5675,265.173913,0.169,2.0,0.4210,25.954076,36.008671,...,1.813341,0.0,1.0,89.708,2023,Sakhir,1,1,NaN,NaN
1,1141,2,92.749,99.9630,292.673077,1.842,20.0,1.8165,25.954076,36.008671,...,1.813341,0.0,16.0,91.652,2023,Sakhir,0,0,NaN,NaN
2,1141,4,91.570,99.7190,276.440000,0.663,9.0,1.5725,25.954076,36.008671,...,1.813341,0.0,11.0,91.381,2023,Sakhir,0,0,NaN,NaN
3,1141,10,91.475,100.0480,290.640000,0.568,7.0,1.9015,25.954076,36.008671,...,1.813341,0.0,20.0,92.181,2023,Sakhir,0,0,NaN,NaN
4,1141,11,91.078,98.2030,272.590909,0.171,3.0,0.0565,25.954076,36.008671,...,1.813341,0.0,2.0,89.846,2023,Sakhir,1,1,NaN,NaN


## 1. Cek data & label

Yang diperiksa:

- **Label `grid_position` harus terisi** — baris kosong = pembalap yang tidak menetapkan
  waktu kualifikasi, harus dibuang karena tidak punya target.
- Missing value pada fitur.
- Distribusi `grid_position` (harusnya 1–20, tiap posisi muncul sekali per weekend).

In [2]:
TARGET = "grid_position"

print("=== Missing value per kolom ===")
missing = df.isna().sum()
print(missing[missing > 0] if (missing > 0).any() else "Tidak ada missing")
print()

print("Baris tanpa label grid_position:", df[TARGET].isna().sum())

# Buang baris tanpa target (pembalap tanpa waktu kualifikasi)
before = len(df)
df = df[df[TARGET].notna()].copy()
df[TARGET] = df[TARGET].astype(int)
print(f"Drop baris tanpa label: {before} -> {len(df)}")
print()

print("=== Distribusi grid_position ===")
print(df[TARGET].value_counts().sort_index().to_string())

=== Missing value per kolom ===
grid_position         140
quali_lap_duration    202
hist_grid_season      123
hist_n                 53
dtype: int64

Baris tanpa label grid_position: 140
Drop baris tanpa label: 1974 -> 1834

=== Distribusi grid_position ===
grid_position
1     90
2     90
3     90
4     90
5     90
6     89
7     90
8     90
9     90
10    89
11    90
12    90
13    90
14    90
15    90
16    90
17    90
18    90
19    89
20    87
21    20
22    20


## 2. EDA — fitur mana yang paling berkorelasi dengan grid?

Menghitung korelasi Spearman tiap fitur terhadap `grid_position`.

- **Korelasi positif** (mis. `fp_best_gap`): makin jauh gap ke tercepat → makin besar angka grid (lebih belakang) = hubungan yang diharapkan.
- Fitur dengan korelasi besar terhadap target adalah kandidat fitur paling berguna.
- Perbandingan penting: korelasi angka absolut `fp_best_lap` yang digabung semua sirkuit hanya **0.061**
  (nyaris nol), sedangkan diukur per weekend mencapai **0.657** — inilah alasan fitur diubah ke gap/rank.

In [3]:
FEATURES = [
    # --- relatif dalam weekend (bahasa yang sama dengan target) ---
    "fp_best_gap",      # selisih detik ke tercepat di weekend yang sama
    "fp_best_rank",     # peringkat dalam weekend = informasi yang dipakai baseline
    "fp_median_gap",    # selisih pace konsisten (median) ke tercepat
    # --- histori (hanya data masa lalu; menangkap skill pembalap lintas weekend) ---
    "hist_grid_season", # rata-rata grid pembalap di weekend sebelumnya
    "hist_n",           # jumlah weekend terlewati (pengalaman)
]

# Kenapa daftarnya begini (lihat bagian 1b & 1c)?
# - 3 fitur pertama RELATIF per weekend → skalanya sama di semua sirkuit
#   (rank 1-20, gap 0-3 detik), berbeda dengan angka absolut yang
#   bervariasi 39 detik antar sirkuit.
# - 2 fitur terakhir HISTORI → menjawab keterbatasan fitur FP: performa
#   pembalap/mobil yang stabil tidak terlihat dari 1 weekend latihan saja.
# - Fitur absolut (fp_best_lap, fp_median_clean_lap, fp_avg_st_speed) dibuang:
#   korelasi pool-nya nyaris nol (0.061 / 0.077 / -0.055).
# - Fitur cuaca dibuang: nilainya SAMA untuk semua 20 pembalap dalam 1 weekend
#   → mustahil menjelaskan urutan grid (korelasi ~0).

corr = (
    df[FEATURES + [TARGET]]
    .corr(method="spearman")[TARGET]
    .drop(TARGET)
    .sort_values()
)

print("Korelasi Spearman dengan grid_position:")
print(corr.round(3).to_string())

df[FEATURES].describe().round(3)

Korelasi Spearman dengan grid_position:
hist_n             -0.029
fp_median_gap       0.220
fp_best_gap         0.588
hist_grid_season    0.631
fp_best_rank        0.669


,fp_best_gap,fp_best_rank,fp_median_gap,hist_grid_season,hist_n
count,1834.000,1834.000,1834.000,1800.000,1804.000
mean,1.537,10.774,4.934,10.382,36.896
std,3.508,6.017,5.848,3.912,24.371
min,0.000,1.000,0.000,1.000,0.000
25%,0.556,6.000,1.101,6.874,16.000
50%,1.002,11.000,2.563,11.628,34.000
75%,1.506,16.000,6.993,13.348,56.000
max,44.806,27.000,47.772,20.000,89.000


## 3. Baseline — pembanding tanpa machine learning

Aturan baseline: **urutkan pembalap menurut `fp_best_lap` tercepat per weekend**,
lalu angka urutannya jadi prediksi grid.

Baseline dievaluasi dengan **fungsi metrik yang sama** dan **weekend test yang sama**
dengan model. Sejak revisi ini, metrik dihitung **per weekend** lalu dirata-ratakan.

In [4]:
def baseline_predict(test_df):
    """Prediksi grid = ranking fp_best_lap PER weekend (tercepat = 1)."""
    pred = pd.Series(pd.NA, index=test_df.index, dtype="Int64")
    for _, group in test_df.groupby("meeting_key"):
        ranked = group.sort_values("fp_best_lap")
        pred.loc[ranked.index] = np.arange(1, len(ranked) + 1)
    return pred


METRIC_COLS = ["mae", "spearman", "kendall_tau", "ndcg_at_3"]


def rank_predictions(test_df, pred_score):
    """Skor regresi -> posisi UNIK 1..N per weekend (tie-break deterministik).

    PERBAIKAN: dulu `np.rint(pred)` + clip ke `len(test)` (seluruh tahun!) bisa
    menghasilkan dua pembalap dengan posisi sama. Sekarang di-rank per meeting_key.
    Skor lebih kecil = grid lebih depan.
    """
    out = pd.DataFrame({
        "_row": np.arange(len(test_df)),
        "meeting_key": test_df["meeting_key"].to_numpy(),
        "driver_number": test_df["driver_number"].to_numpy(),
        "pred_score": np.asarray(pred_score, dtype=float),
    })
    out = out.sort_values(["meeting_key", "pred_score", "driver_number"])
    out["y_pred_rank"] = (
        out.groupby("meeting_key")["pred_score"]
        .rank(method="first", ascending=True).astype(int)
    )
    return out.sort_values("_row")["y_pred_rank"].to_numpy()


def evaluate_by_weekend(test_df, y_true, y_pred_rank):
    """Metrik dihitung PER WEEKEND lalu dirata-ratakan (tidak di-pool per tahun).

    Spearman yang di-pool lintas weekend menyesatkan: posisi 1 di Monza bukan
    satu kelompok ranking dengan posisi 1 di Monaco.
    """
    tmp = test_df[["meeting_key", "year"]].copy()
    tmp["y_true"] = np.asarray(y_true, dtype=float)
    tmp["y_pred"] = np.asarray(y_pred_rank, dtype=float)

    rows = []
    for meeting_key, group in tmp.groupby("meeting_key"):
        yt = group["y_true"].to_numpy()
        yp = group["y_pred"].to_numpy()
        if np.std(yt) > 0 and np.std(yp) > 0:
            rho = spearmanr(yt, yp).statistic
            tau = kendalltau(yt, yp).statistic
        else:
            rho = tau = np.nan

        # NDCG@3: posisi asli lebih depan -> relevansi lebih tinggi
        relevance = (yt.max() + 1 - yt).reshape(1, -1)
        score = (-yp).reshape(1, -1)
        ndcg3 = ndcg_score(relevance, score, k=3)

        rows.append({
            "meeting_key": meeting_key,
            "year": int(group["year"].iloc[0]),
            "mae": mean_absolute_error(yt, yp),
            "spearman": rho,
            "kendall_tau": tau,
            "ndcg_at_3": ndcg3,
        })
    return pd.DataFrame(rows)


# Evaluasi baseline pada seluruh data (per weekend)
baseline_rows = []
for _, group in df.groupby("meeting_key"):
    pred = baseline_predict(group)
    baseline_rows.append(
        evaluate_by_weekend(group, group[TARGET], pred).iloc[0].to_dict()
    )

baseline_eval = pd.DataFrame(baseline_rows)
print("=== Baseline (ranking fp_best_lap; metrik rata-rata per weekend) ===")
print(baseline_eval[METRIC_COLS].mean().round(3).to_string())

=== Baseline (ranking fp_best_lap; metrik rata-rata per weekend) ===
mae            3.526
spearman       0.657
kendall_tau    0.509
ndcg_at_3      0.878


## 4. Split — expanding-window time split

Menghindari *leakage temporal* (model tidak boleh "melihat masa depan"):

| Test year | Train |
|---|---|
| 2024 | 2023 |
| 2025 | 2023–2024 |
| 2026 | 2023–2025 |

Kalau data baru punya 1 tahun saja, split time-based tidak bisa dibuat —
notebook akan memberi peringatan dan fallback ke K-Fold (dengan catatan: hasilnya
belum benar-benar bebas leakage, hanya untuk uji coba).

In [5]:
def expanding_split(data):
    """Bagi data menjadi split train/test berbasis tahun (expanding window)."""
    years = sorted(data["year"].unique())
    splits = []
    for i in range(1, len(years)):
        train_years = years[:i]
        test_year = years[i]
        train = data[data["year"].isin(train_years)]
        test = data[data["year"] == test_year]
        if len(train) and len(test):
            splits.append((train_years, test_year, train, test))
    return splits


splits = expanding_split(df)

if splits:
    print("Expanding-window split:")
    for train_years, test_year, train, test in splits:
        print(f"  test {test_year}  <-  train {train_years}  ({len(train)} train / {len(test)} test)")
else:
    print("PERINGATAN: data baru punya 1 tahun — split time-based tidak tersedia.")
    print("Model hanya akan diuji dengan K-Fold (belum bebas leakage temporal).")

Expanding-window split:
  test 2024  <-  train [np.int64(2023)]  (436 train / 479 test)
  test 2025  <-  train [np.int64(2023), np.int64(2024)]  (915 train / 480 test)
  test 2026  <-  train [np.int64(2023), np.int64(2024), np.int64(2025)]  (1395 train / 439 test)


## 5. Training — 4 kandidat model, satu target yang sama

`alpha` Ridge dan pilihan model ditentukan di **validasi** (4b), bukan di test.
Keempat kandidat: Ridge, RandomForest, HistGradientBoosting, **XGBoost**.

Prediksi skor diubah jadi **peringkat unik per `meeting_key`** — bukan dibulatkan
independen seperti sebelumnya.

## 4b. Tuning — pilih alpha Ridge & model di VALIDASI (bukan test)

`alpha=1.0` dan pilihan model **tidak boleh** ditentukan dari test year.
Kandidat alpha `[0.001 … 100]` diuji dengan expanding-window:

```text
train 2023 -> validasi 2024
train 2023–2024 -> validasi 2025
```

**2026 disisihkan sebagai holdout final** — dipakai sekali di bagian evaluasi.
Metrik dihitung per weekend, lalu dirata-ratakan.

In [6]:
def build_pipeline(model_name, alpha=1.0):
    """Pipeline: imputasi (+ scaling untuk model linear) + estimator."""
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if model_name == "Ridge":
        numeric_steps.append(("scaler", StandardScaler()))

    preprocessor = ColumnTransformer(
        [("num", Pipeline(numeric_steps), FEATURES)],
        remainder="drop",
    )

    if model_name == "Ridge":
        estimator = Ridge(alpha=alpha, random_state=SEED)
    elif model_name == "RandomForest":
        estimator = RandomForestRegressor(
            n_estimators=300, random_state=SEED, n_jobs=-1
        )
    elif model_name == "HistGB":
        estimator = HistGradientBoostingRegressor(random_state=SEED)
    elif model_name == "XGBoost":
        estimator = XGBRegressor(
            n_estimators=300, max_depth=3, learning_rate=0.03,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=5.0,
            objective="reg:squarederror", random_state=SEED, n_jobs=-1,
        )
    else:
        raise ValueError(model_name)

    return Pipeline([
        ("prep", preprocessor),
        ("model", estimator),
    ])


RIDGE_ALPHAS = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
MODEL_NAMES = ["Ridge", "RandomForest", "HistGB", "XGBoost"]
ALL_YEARS = sorted(df["year"].unique())
VAL_SPLITS = [(ALL_YEARS[:i], ALL_YEARS[i]) for i in (1, 2)]  # 2024, 2025


def fit_and_score(name, train, test, alpha=1.0):
    pipe = build_pipeline(name, alpha=alpha)
    pipe.fit(train[FEATURES], train[TARGET])
    return pipe.predict(test[FEATURES])


def mean_weekend_metrics(test, score):
    rank = rank_predictions(test, score)
    return evaluate_by_weekend(test, test[TARGET], rank)[METRIC_COLS].mean()


# --- (a) cari alpha terbaik di validasi ---
tuning_rows = []
for train_years, val_year in VAL_SPLITS:
    tr = df[df["year"].isin(train_years)]
    va = df[df["year"] == val_year]
    for alpha in RIDGE_ALPHAS:
        m = mean_weekend_metrics(va, fit_and_score("Ridge", tr, va, alpha=alpha))
        tuning_rows.append({"alpha": alpha, "val_year": val_year, **m.to_dict()})

tuning_results = pd.DataFrame(tuning_rows)
alpha_summary = tuning_results.groupby("alpha")[METRIC_COLS].mean()
best_alpha = float(alpha_summary["mae"].idxmin())

# --- (b) pilih model terbaik di validasi (dengan alpha Ridge sudah tetap) ---
validation_rows = []
for train_years, val_year in VAL_SPLITS:
    tr = df[df["year"].isin(train_years)]
    va = df[df["year"] == val_year]
    for name in MODEL_NAMES:
        m = mean_weekend_metrics(va, fit_and_score(name, tr, va, alpha=best_alpha))
        validation_rows.append({"model": name, "val_year": val_year, **m.to_dict()})

validation_results = pd.DataFrame(validation_rows)
best_model = validation_results.groupby("model")["mae"].mean().idxmin()

print("=== Tuning alpha Ridge (validasi) ===")
print(alpha_summary.round(3).to_string())
print(f"alpha terpilih: {best_alpha}\n")
print("=== Perbandingan model di validasi ===")
print(validation_results.groupby("model")[METRIC_COLS].mean().round(3).sort_values("mae").to_string())
print(f"\nModel terpilih (VALIDASI): {best_model}")

=== Tuning alpha Ridge (validasi) ===
           mae  spearman  kendall_tau  ndcg_at_3
alpha                                           
0.001    3.254     0.718        0.546      0.918
0.010    3.254     0.718        0.546      0.918
0.100    3.254     0.718        0.546      0.918
1.000    3.254     0.718        0.546      0.918
10.000   3.254     0.718        0.546      0.918
100.000  3.261     0.717        0.544      0.920
alpha terpilih: 0.001

=== Perbandingan model di validasi ===
                mae  spearman  kendall_tau  ndcg_at_3
model                                                
Ridge         3.254     0.718        0.546      0.918
XGBoost       3.357     0.696        0.530      0.888
RandomForest  3.661     0.647        0.485      0.880
HistGB        3.673     0.640        0.479      0.883

Model terpilih (VALIDASI): Ridge


In [7]:
if splits:
    metric_rows = []
    prediction_rows = []
    weekend_rows = []

    for train_years, test_year, train, test in splits:
        # Baseline — evaluasi dengan fungsi & weekend yang sama
        base_pred = baseline_predict(test)
        wm = evaluate_by_weekend(test, test[TARGET], base_pred)
        metric_rows.append({
            "model": "Baseline", "test_year": test_year,
            **wm[METRIC_COLS].mean().to_dict(),
        })
        weekend_rows.append(wm.assign(model="Baseline", test_year=test_year))

        for name in MODEL_NAMES:
            pipe = build_pipeline(name, alpha=best_alpha)
            pipe.fit(train[FEATURES], train[TARGET])
            pred_score = pipe.predict(test[FEATURES])
            pred_rank = rank_predictions(test, pred_score)   # unik per weekend

            wm = evaluate_by_weekend(test, test[TARGET], pred_rank)
            metric_rows.append({
                "model": name, "test_year": test_year,
                **wm[METRIC_COLS].mean().to_dict(),
            })
            weekend_rows.append(wm.assign(model=name, test_year=test_year))

            for pos, (_, row) in enumerate(test.iterrows()):
                prediction_rows.append({
                    "model": name,
                    "test_year": test_year,
                    "meeting_key": row["meeting_key"],
                    "driver_number": row["driver_number"],
                    "y_true": int(row[TARGET]),
                    "pred_score": float(pred_score[pos]),
                    "y_pred": int(pred_rank[pos]),
                })

    metrics = pd.DataFrame(metric_rows)
    predictions = pd.DataFrame(prediction_rows)
    weekend_metrics = pd.concat(weekend_rows, ignore_index=True)

    print("=== Metrik rata-rata per weekend, per model per test year ===")
    print(metrics.round(3).to_string(index=False))
    print()
    print("=== Rata-rata across test year ===")
    print(metrics.groupby("model")[METRIC_COLS].mean().round(3).sort_values("mae").to_string())
else:
    metrics = pd.DataFrame()
    predictions = pd.DataFrame()
    weekend_metrics = pd.DataFrame()
    print("Belum ada split — data masih 1 tahun. Tidak bisa training time-based.")

=== Metrik rata-rata per weekend, per model per test year ===
       model  test_year   mae  spearman  kendall_tau  ndcg_at_3
    Baseline       2024 3.442     0.669        0.515      0.854
       Ridge       2024 3.146     0.729        0.560      0.913
RandomForest       2024 3.306     0.697        0.531      0.877
      HistGB       2024 3.501     0.656        0.497      0.859
     XGBoost       2024 3.301     0.698        0.535      0.860
    Baseline       2025 3.604     0.647        0.487      0.901
       Ridge       2025 3.363     0.706        0.531      0.924
RandomForest       2025 4.017     0.597        0.439      0.883
      HistGB       2025 3.846     0.623        0.461      0.907
     XGBoost       2025 3.412     0.694        0.525      0.915
    Baseline       2026 2.770     0.800        0.658      0.895
       Ridge       2026 3.201     0.767        0.608      0.881
RandomForest       2026 4.282     0.612        0.441      0.867
      HistGB       2026 4.195     0.622   

## 6. Evaluasi — apakah model mengalahkan baseline?

Metrik dihitung **per weekend**, lalu dirata-ratakan: **MAE** (selisih posisi),
**Spearman** & **Kendall's Tau** (kecocokan urutan), **NDCG@3** (kualitas 3 teratas).

Model terpilih berdasarkan validasi; **2026 disisihkan sebagai holdout final**.
Bandingkan selalu dengan baseline pada weekend yang sama.

In [8]:
if len(metrics):
    summary = metrics.groupby("model")[METRIC_COLS].mean().sort_values("mae")
    base = summary.loc["Baseline"]

    print(f"Model terpilih di VALIDASI: {best_model} (alpha Ridge: {best_alpha})")
    print(f"Holdout final: {ALL_YEARS[-1]} — belum dipakai untuk tuning\n")
    print("Perbandingan dengan baseline (rata-rata metrik per weekend):")
    print()
    for model_name, row in summary.iterrows():
        if model_name == "Baseline":
            tag = "(pembanding)"
        elif row["mae"] < base["mae"] and row["spearman"] > base["spearman"]:
            tag = "✅ MENANG dua metrik"
        elif row["mae"] < base["mae"]:
            tag = "✅ MAE lebih baik, Spearman tidak"
        else:
            tag = "❌ kalah dari baseline"
        print(
            f"{model_name:15s} MAE {row['mae']:.3f}  Spearman {row['spearman']:.3f}"
            f"  Kendall {row['kendall_tau']:.3f}  NDCG@3 {row['ndcg_at_3']:.3f}  {tag}"
        )

    print()
    sel = summary.loc[best_model]
    holdout = metrics[metrics["test_year"] == ALL_YEARS[-1]].set_index("model")
    print(f"=== Model terpilih ({best_model}) pada holdout {ALL_YEARS[-1]} ===")
    print(f"MAE {holdout.loc[best_model, 'mae']:.3f} (baseline {holdout.loc['Baseline', 'mae']:.3f})")
    print(f"Spearman {holdout.loc[best_model, 'spearman']:.3f} "
          f"(baseline {holdout.loc['Baseline', 'spearman']:.3f})")
else:
    print("Belum ada metrik untuk dievaluasi.")

Model terpilih di VALIDASI: Ridge (alpha Ridge: 0.001)
Holdout final: 2026 — belum dipakai untuk tuning

Perbandingan dengan baseline (rata-rata metrik per weekend):

Ridge           MAE 3.237  Spearman 0.734  Kendall 0.566  NDCG@3 0.906  ✅ MENANG dua metrik
Baseline        MAE 3.272  Spearman 0.705  Kendall 0.553  NDCG@3 0.883  (pembanding)
XGBoost         MAE 3.600  Spearman 0.680  Kendall 0.513  NDCG@3 0.886  ❌ kalah dari baseline
HistGB          MAE 3.847  Spearman 0.634  Kendall 0.469  NDCG@3 0.872  ❌ kalah dari baseline
RandomForest    MAE 3.868  Spearman 0.636  Kendall 0.471  NDCG@3 0.876  ❌ kalah dari baseline

=== Model terpilih (Ridge) pada holdout 2026 ===
MAE 3.201 (baseline 2.770)
Spearman 0.767 (baseline 0.800)


## 7. Turunan top-3 & top-10 (bukan model terpisah)

`is_top3` / `is_top10` diturunkan dari prediksi grid. ROC-AUC memakai
**skor kontinu (`pred_score`)** sebelum di-rank, dihitung **per weekend**, lalu dirata-ratakan.

In [9]:
if len(predictions):
    topk_rows = []
    for model_name, group in predictions.groupby("model"):
        for k, col in ((3, "is_top3"), (10, "is_top10")):
            aucs, hits = [], []
            for (_, _), g in group.groupby(["test_year", "meeting_key"]):
                yt = (g["y_true"] <= k).astype(int)
                if yt.nunique() < 2:
                    continue
                # ROC-AUC pakai SKOR KONTINU sebelum ranking
                aucs.append(roc_auc_score(yt, -g["pred_score"].to_numpy()))
                yp = (g["y_pred"] <= k).astype(int)
                hits.append((yt.to_numpy() == yp.to_numpy()).mean())
            topk_rows.append({
                "model": model_name,
                "target": col,
                "roc_auc": float(np.nanmean(aucs)) if aucs else np.nan,
                "accuracy": float(np.nanmean(hits)) if hits else np.nan,
            })

    topk = pd.DataFrame(topk_rows)
    print("=== Top-3 / Top-10 (AUC skor kontinu, rata-rata per weekend) ===")
    print(topk.round(3).to_string(index=False))
else:
    topk = pd.DataFrame()
    print("Belum ada prediksi untuk dihitung top-k.")

=== Top-3 / Top-10 (AUC skor kontinu, rata-rata per weekend) ===
       model   target  roc_auc  accuracy
      HistGB  is_top3    0.869     0.856
      HistGB is_top10    0.822     0.755
RandomForest  is_top3    0.874     0.863
RandomForest is_top10    0.826     0.753
       Ridge  is_top3    0.896     0.867
       Ridge is_top10    0.879     0.797
     XGBoost  is_top3    0.889     0.864
     XGBoost is_top10    0.849     0.769


## 8. Feature importance — fitur apa yang paling menentukan grid?

Memakai **permutation importance** pada model terbaik (model-agnostic, bisa dipakai
untuk Ridge maupun tree). Fitur yang kalau diacak nilainya membuat prediksi jauh
lebih buruk = fitur penting.

In [10]:
from sklearn.inspection import permutation_importance

if len(metrics):
    # Pakai split terakhir (data train terbanyak) untuk mengukur importance
    train_years, test_year, train, test = splits[-1]

    pipe = build_pipeline(best_model)
    pipe.fit(train[FEATURES], train[TARGET])

    result = permutation_importance(
        pipe, test[FEATURES], test[TARGET],
        n_repeats=30, random_state=SEED, scoring="neg_mean_absolute_error",
    )

    importance = pd.DataFrame({
        "fitur": FEATURES,
        "importance_mean": result.importances_mean,
        "importance_std": result.importances_std,
    }).sort_values("importance_mean", ascending=False)

    print(f"Permutation importance — model {best_model} (test {test_year}):\n")
    print(importance.round(4).to_string(index=False))
else:
    importance = pd.DataFrame()
    print("Belum ada model untuk diukur importance-nya.")

Permutation importance — model Ridge (test 2026):

           fitur  importance_mean  importance_std
    fp_best_rank           1.5529          0.1123
hist_grid_season           1.0214          0.0983
   fp_median_gap           0.0497          0.0146
          hist_n           0.0433          0.0101
     fp_best_gap          -0.0026          0.0016


File yang ditulis ke `outputs/`:

| File | Isi |
|---|---|
| `model_metrics.csv` | MAE, Spearman, Kendall, NDCG@3 per model per test year |
| `model_summary.csv` | rata-rata metrik antar-test year |
| `weekend_metrics.csv` | metrik per weekend (bisa diaudit per sirkuit) |
| `predictions.csv` | `pred_score` (kontinu) + `y_pred` (ranking unik) + `y_true` |
| `topk_metrics.csv` | ROC-AUC & accuracy top-3/top-10 per weekend |
| `feature_importance.csv` | permutation importance |
| `tuning_results.csv` | hasil pencarian alpha Ridge di validasi |
| `experiment_config.json` | fitur, seed, tahun train/test, alpha & model terpilih |
| `laporan.md` | ringkasan naratif hasil |

In [11]:
out_dir = resolve_path(cfg["paths"]["outputs"])
out_dir.mkdir(parents=True, exist_ok=True)

saved = []

if len(metrics):
    metrics.to_csv(out_dir / "model_metrics.csv", index=False)
    saved.append("model_metrics.csv")
    metrics.groupby("model")[METRIC_COLS].mean().sort_values("mae").to_csv(
        out_dir / "model_summary.csv")
    saved.append("model_summary.csv")
if len(weekend_metrics):
    weekend_metrics.to_csv(out_dir / "weekend_metrics.csv", index=False)
    saved.append("weekend_metrics.csv")
if len(predictions):
    predictions.to_csv(out_dir / "predictions.csv", index=False)
    saved.append("predictions.csv")
if len(topk):
    topk.to_csv(out_dir / "topk_metrics.csv", index=False)
    saved.append("topk_metrics.csv")
if len(importance):
    importance.to_csv(out_dir / "feature_importance.csv", index=False)
    saved.append("feature_importance.csv")
if len(tuning_results):
    tuning_results.to_csv(out_dir / "tuning_results.csv", index=False)
    saved.append("tuning_results.csv")

experiment_config = {
    "features": FEATURES,
    "target": TARGET,
    "seed": SEED,
    "years": [int(y) for y in ALL_YEARS],
    "train_test_splits": [
        {"train": [int(y) for y in ty], "test_year": int(ty2)}
        for ty, ty2, _, _ in splits
    ],
    "ridge_alphas": RIDGE_ALPHAS,
    "selected_alpha": best_alpha,
    "selected_model_validation": best_model,
    "models_compared": MODEL_NAMES,
    "ranking": "rank unik per meeting_key (method=first)",
    "evaluation": "metrik per weekend, lalu rata-rata",
}
(out_dir / "experiment_config.json").write_text(
    json.dumps(experiment_config, indent=2), encoding="utf-8")
saved.append("experiment_config.json")

# Laporan singkat
lines = [
    "# Laporan Modeling — Prediksi Grid Kualifikasi",
    "",
    f"- Jumlah baris data: {len(df)}",
    f"- Tahun: {[int(y) for y in ALL_YEARS]}",
    f"- Weekend unik: {df['meeting_key'].nunique()}",
    f"- Fitur: {', '.join(FEATURES)}",
    f"- Target: {TARGET}",
    f"- Alpha Ridge terpilih (validasi): {best_alpha}",
    f"- Model terpilih (validasi): {best_model}",
    f"- Holdout final: {int(ALL_YEARS[-1])}",
    f"- Evaluasi: metrik per weekend lalu dirata-ratakan; ranking unik per meeting_key",
    "",
]

if len(metrics):
    summary = metrics.groupby("model")[METRIC_COLS].mean().sort_values("mae")
    lines.append("## Rata-rata metrik per weekend")
    lines.append("")
    lines.append("| Model | MAE | Spearman | Kendall | NDCG@3 |")
    lines.append("|---|---|---|---|---|")
    for model_name, row in summary.iterrows():
        lines.append(
            f"| {model_name} | {row['mae']:.3f} | {row['spearman']:.3f} | "
            f"{row['kendall_tau']:.3f} | {row['ndcg_at_3']:.3f} |"
        )
    lines.append("")
    base_mae = summary.loc["Baseline", "mae"]
    winner = summary.drop("Baseline").index[0]
    lines.append(f"MAE terbaik: **{winner}** ({summary.loc[winner, 'mae']:.3f} "
                 f"vs baseline {base_mae:.3f})")
    lines.append(f"Model terpilih berdasarkan VALIDASI: **{best_model}**")
else:
    lines.append("Data belum cukup untuk training time-based (butuh ≥ 2 tahun).")

report_path = out_dir / "laporan.md"
report_path.write_text("\n".join(lines), encoding="utf-8")
saved.append("laporan.md")

print("Saved:")
for name in saved:
    print(" -", out_dir / name)

Saved:
 - C:\syntax\codelab\SubrisetData\prediksi-openf1\outputs\model_metrics.csv
 - C:\syntax\codelab\SubrisetData\prediksi-openf1\outputs\model_summary.csv
 - C:\syntax\codelab\SubrisetData\prediksi-openf1\outputs\weekend_metrics.csv
 - C:\syntax\codelab\SubrisetData\prediksi-openf1\outputs\predictions.csv
 - C:\syntax\codelab\SubrisetData\prediksi-openf1\outputs\topk_metrics.csv
 - C:\syntax\codelab\SubrisetData\prediksi-openf1\outputs\feature_importance.csv
 - C:\syntax\codelab\SubrisetData\prediksi-openf1\outputs\tuning_results.csv
 - C:\syntax\codelab\SubrisetData\prediksi-openf1\outputs\experiment_config.json
 - C:\syntax\codelab\SubrisetData\prediksi-openf1\outputs\laporan.md


## Ringkasan output `02_model.ipynb`

| Output | Keterangan |
|---|---|
| Tabel metrik | MAE, Spearman, Kendall & NDCG@3 per model vs baseline |
| Tabel weekend | metrik per weekend + hasil tuning alpha Ridge |
| Tabel top-k | ROC-AUC top-3 / top-10 (turunan dari grid) |
| Feature importance | fitur FP paling menentukan grid |
| `outputs/*.csv` + `outputs/laporan.md` | hasil tersimpan untuk laporan akhir |

**Hasil sekarang:** Ridge **mengalahkan baseline** (MAE rata-rata per weekend
3.237 vs 3.272; Spearman 0.734 vs 0.705) berkat dua perubahan fitur: (1) angka absolut →
gap/rank per weekend, (2) tambahan fitur histori grid pembalap. Alpha dan model
dipilih di validasi; 2026 disisihkan sebagai holdout final.

**Prinsipnya tetap:** kalau model tidak mengalahkan baseline, itu temuan yang sah —
artinya fiturnya belum cukup. Solusinya menambah data/informasi
(lebih banyak weekend, fitur histori lebih dalam),
bukan memaksa model lebih canggih.